# Formal Semantic Parsing with Language Models

Some useful functions and some functions that you need to implement are presented in this notebook. You don't have to use them though.

In this assignment, we will evaluate the ability of language on semantic parsing task. In particular, SQL parsing. The assignment has three parts:


1.   Basic Prompt
2.   Finetuning
3.   Context-Free Grammer

In each parts, you will exaime the model output in terms of correctness and output well-formedness.

Part 2 and 3 can be replaced by other means such as RAG(Retrieval-Augmented Generation). Students are wellcomed to propose their own ideas to solve this task.

References:
https://github.com/jkkummerfeld/text2sql-data/

https://github.com/mlc-ai/xgrammar

**This starting code is based on old transformers-cfg, you have to use xgrammar. You need to check the official document of xgrammar or Lab7. There are some functions for you to fill to**

**You can run small model on Kaggle notebook https://www.kaggle.com/ for free**



In [ ]:
!pip install transformers datasets trl peft
!pip install xgrammar

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.5/465.5 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 13.1 MB/s eta 0:00:00


In [ ]:
!wget https://raw.githubusercontent.com/jkkummerfeld/text2sql-data/master/data/geography-db.added-in-2020.sqlite
!wget https://raw.githubusercontent.com/jkkummerfeld/text2sql-data/master/data/geography-fields.txt
!wget https://raw.githubusercontent.com/jkkummerfeld/text2sql-data/master/data/geography-schema.csv
!wget https://raw.githubusercontent.com/jkkummerfeld/text2sql-data/master/data/geography.json
!wget https://raw.githubusercontent.com/jkkummerfeld/text2sql-data/master/data/geography-db.sql

--2025-12-03 06:07:52--  https://raw.githubusercontent.com/jkkummerfeld/text2sql-data/master/data/geography-db.added-in-2020.sqlite
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 65536 (64K) [application/octet-stream]
Saving to: ‘geography-db.added-in-2020.sqlite’

geography-db.added- 100%[===================>]  64.00K  --.-KB/s    in 0.01s   

2025-12-03 06:07:52 (4.37 MB/s) - ‘geography-db.added-in-2020.sqlite’ saved [65536/65536]

--2025-12-03 06:07:52--  https://raw.githubusercontent.com/jkkummerfeld/text2sql-data/master/data/geography-fields.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443

In [ ]:
import json

def extract_sentence_fields(sentence):
    text = sentence["text"]
    variables = sentence["variables"]
    split = sentence["question-split"]
    return text, variables, split

def insert_variables(sql, sql_variables, sent, sent_variables):
    for info in sql_variables:
        name = info['name']
        value = info['example']
        if name in sent_variables and sent_variables[name] != "":
            value = sent_variables[name]
        sent = value.join(sent.split(name))
        qvalue = '{}'.format(value)
        sql = qvalue.join(sql.split(name))
    return (sql, sent)
def build_question_split(jsons,making_prompt=lambda x:x, keep_variables=False):
    datasets = {}
    for json_dict in jsons:
        for query in [json_dict["sql"][0]]:
            sql_vars = json_dict['variables']
            for sentence in json_dict["sentences"]:
                text, variables, split = extract_sentence_fields(sentence)
                if split == "exclude":
                    continue
                if keep_variables:
                    sql = query
                    question = text
                else:
                    sql, question = insert_variables(
                        query, sql_vars, text, variables)
           #     sql = tokenise(sql)
            #    question = preprocess_text(question)
                if not split in datasets:
                    datasets[split] = []
                example = {}
                example["text"] = making_prompt(question)+sql
                example["question"] = question
                example["sql"] = sql
                datasets[split].append(example)
    return datasets

making_prompt = lambda x:x
with open("geography.json", 'r') as file:
    geography_data = json.load(file)
    geography_datasets =  build_question_split(geography_data,making_prompt=making_prompt)

In [ ]:
import sqlite3
def load_sqlite_file(file_path):
    """
    Load a .sqlite file and return a connection object.

    Args:
        file_path (str): Path to the .sqlite file

    Returns:
        sqlite3.Connection: Connection object to the loaded database
    """
    try:
        # Establish a connection to the database
        conn = sqlite3.connect(file_path)
        print(f"Loaded database from {file_path}")
        return conn
    except sqlite3.Error as e:
        print(f"Error loading database: {e}")
        return None

def get_all_results(dataset,cursor):
    for example in dataset:
        question = example["question"]
        sql = example["sql"]
        cursor.execute(sql)
        gold_answers = cursor.fetchall()
        example["answers"] = gold_answers

file_path = "/content/geography-db.added-in-2020.sqlite"  # Replace with your .sqlite file path
conn = load_sqlite_file(file_path)
cursor = conn.cursor()
for dataset in geography_datasets:
    get_all_results(dataset,cursor)



# Don't forget to close the connection when you're done
conn.close()

#get true positive, true negative, false positive, false negative and where exact match
def compare_results(generated,answers):
  generated=
  answers=



SyntaxError: invalid syntax (ipython-input-3036837747.py, line 42)

In [ ]:
#need to fill, should return micro/marco precision, recall, f1
# and exact match, grammatical_ratio
def evaluate(dataset,model,conn, tokenizer,making_prompt=lambda x:x,cfg=None,cfg_enbf=None):
  # Below is from transformer-cfg, you need to change them into XGrammar compatible
    if cfg is not None:
        grammar_processor = GrammarConstrainedLogitsProcessor(cfg)
    elif cfg_enbf is not None:
        # Load JSON grammar
        with open(cfg_enbf, "r") as file:
            grammar_str = file.read()
        grammar = IncrementalGrammarConstraint(grammar_str, "root", tokenizer)
        grammar_processor = GrammarConstrainedLogitsProcessor(grammar)
        grammar_processors = []
    else:
        grammar_processors = None

    correct = 0
    total = 0
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    for example in dataset:
        question = example["question"]
   #     sql = example["sql"]
        gold_answers = example["answers"]
        prompt = making_prompt(question)
        # Generate
        inputs = tokenizer(prompt, return_tensors='pt').to(device)
        generation_config = GenerationConfig(
            num_beams=5,
            no_repeat_ngram_size=2,
            temperature=0.7,
            top_k=50,
            top_p=0.9,
            do_sample=True
        )
        output = model.generate(
            **inputs,
          # max_length=50,
         #   generation_config=generation_config,
            max_new_tokens=512,
            logits_processor=grammar_processors,
            repetition_penalty=1.1,
            num_return_sequences=1,
        )
        # Decode output
        generations = tokenizer.batch_decode(output, skip_special_tokens=True)

        cursor.execute(sql)
        generated = cursor.fetchall()
        tp,tn,fp,fn,exact_match=compare_results(generated,gold_answers)

    return micro_f1,micro_precision,micro_recal, macro_f1,macro_precision,
          macro_recal, exact_match_ratio,grammatical_ratio




In [ ]:
#this is just for reference, we need a sql version
!wget https://github.com/Saibo-creator/transformers-CFG/blob/main/examples/grammars/geo_query.ebnf
#the json grammar is informative to learn how to write the basic elements e.g., strings numbers
!wget https://github.com/Saibo-creator/transformers-CFG/blob/main/examples/grammars/json_minimal.ebnf


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from datasets import Dataset,DatasetDict
from peft import LoraConfig, get_peft_model

from trl import SFTTrainer,SFTConfig
# Load your custom dataset (replace 'csv_file_path' with your actual file path)
import json

train_data =  Dataset.from_list(geography_datasets["train"])
dev_data =  Dataset.from_list(geography_datasets["dev"])
test_data =  Dataset.from_list(geography_datasets["test"])

dataset = DatasetDict({"train": train_data, "dev": dev_data, "test": test_data})
print (dataset["train"][0])
# or for a text file
# dataset = load_dataset('text', data_files={'train': 'train.txt', 'validation': 'valid.txt'})

# Load the model and tokenizer
model_name = "HuggingFaceTB/SmolLM2-360M-Instruct"  # or any other model
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Add a padding token to the tokenizer
tokenizer.pad_token = tokenizer.eos_token # Use eos_token as pad_token

#write your prompts and try to evaluate the model

making_prompt = lambda x:x

In [ ]:
# try figure out how to set up the fine tuning config


# Create the SFT Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["dev"],
    args=sft_config,
    tokenizer=tokenizer, # Pass the tokenizer
)

# Fine-tune the model
trainer.train()

# Save the fine-tuned model
trainer.save_model()

#try evaluate


In [ ]:
#write your grammar in a seperate file following transformer_cfg
#format and test it on your (fine tuned ) model

In [ ]:
#try whatever you want to try/analysis